# ⛓️ Prompt Chaining with a Quality Gate in LangGraph

## Learning Objectives
In this notebook, you will learn:
1. **The prompt chaining pattern** - Decompose one task into a fixed sequence of focused LLM calls
2. **Workflow state** - Carry each step's output forward in a `TypedDict`, with a reducer for a retry counter
3. **Quality gates** - Check an intermediate result in code before it flows downstream
4. **Bounded regeneration** - Loop back to an earlier step at most N times, then stop cleanly
5. **Running and reading the chain** - Watch one topic pass the gate and one get stopped

## Where this fits
Modules 02–04 built agents that decide their own next step. This module starts the **workflow patterns**: *you* fix the sequence of steps in code, and the LLM only fills them in. Prompt chaining is the simplest of them; routing (module 06) and parallelization (module 07) build on the same graph machinery.

## Prerequisites
- LangGraph `StateGraph` basics (nodes, edges, `START`/`END`) — seen in `modules/03_react_agents/notebooks/01_react_agent_langgraph.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env` (no other keys needed)

---
## 📖 Part 1: What is Prompt Chaining?

Prompt chaining represents one of the most fundamental agentic workflow patterns, where complex tasks are decomposed into manageable sequential steps. Each step builds upon the previous one, creating a structured approach that significantly improves accuracy by simplifying individual subtasks before progressing to the next phase.

This pattern excels in scenarios where step-by-step reasoning enhances overall output quality. Rather than overwhelming the language model with a complex, multi-faceted prompt, prompt chaining breaks down the cognitive load into digestible components, allowing for more focused and accurate processing at each stage.

| Property | Value |
|---|---|
| Origin | Anthropic, *Building Effective Agents* (Dec 2024) — [anthropic.com/research/building-effective-agents](https://www.anthropic.com/research/building-effective-agents), "Prompt chaining" section |

The pipeline built here turns a topic into a finished email, with a **gate** after the first step:

```
topic ──▶ extract key points ──▶ [GATE] ──▶ write draft ──▶ polish ──▶ final email
                    ▲               │
                    └── regenerate ─┘  (max 3 attempts, then stop)
```

### How it maps onto LangGraph
1. Define the Task: Start by breaking down the problem into smaller sub-tasks. For example, if you want to generate a detailed report, you might split it into steps like "gather data," "analyze data," and "write summary."

2. Create Nodes: Each sub-task becomes a node in the LangGraph structure. A node could be a prompt that instructs the model to perform a specific action, such as "List key facts about X" or "Summarize the following text."

3. Establish Edges: Edges define the sequence and dependencies between nodes. For instance, the output of the "gather data" node flows into the "analyze data" node, ensuring the model has the necessary context to proceed.

4. Execute the Graph: LangGraph processes the nodes in order, passing information along the edges. The model generates responses step-by-step, refining the output as it progresses through the chain.

5. Iterate if Needed: LangGraph supports conditional logic and loops, so you can revisit earlier nodes or adjust the flow based on intermediate results.

> **Key Insight**: Each LLM call does one small job, so each prompt stays short and each output can be checked before the next call spends tokens on it.

---
## 📦 Step 1: Environment Setup

Load the API key from the project-root `.env`, then create the chat model through the course factory.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

✅ Environment ready


In [2]:
# ============================================================================
# LLM INITIALIZATION: One model for every step of the chain
# ============================================================================
from aied_helpers import get_openai_llm

# temperature > 0 so a regenerated step produces a *different* sample —
# at temperature 0 the regenerate loop would just repeat the same output.
llm = get_openai_llm(model="gpt-4o-mini", temperature=0.7)

print(f"✅ LLM initialized: {llm.model_name} (temperature={llm.temperature})")

✅ LLM initialized: gpt-4o-mini (temperature=0.7)


---
## 🗂️ Part 2: The Workflow State

Every node reads from and writes to one shared state. Each chain step owns one field (`key_points`, `draft_email`, `final_email`), so the state doubles as a record of the whole run.

### Key Concepts:
- **`TypedDict` state**: the schema of what flows through the graph; a node returns only the keys it changed
- **`gate_passed`**: the quality gate's latest verdict, written by one node and read by the router
- **Reducer**: `Annotated[int, operator.add]` tells LangGraph to *add* a node's return value to the field instead of overwriting it — each extraction returns `1`, so `retry_count` counts attempts

In [3]:
# ============================================================================
# STATE: What flows between the chain steps
# ============================================================================
import operator
from typing import Annotated

from typing_extensions import TypedDict


class EmailState(TypedDict):
    """State to track email creation progress."""

    topic: str
    key_points: str
    draft_email: str
    final_email: str
    # operator.add: every extract_key_points() return of 1 is summed (1, 2, 3 …)
    retry_count: Annotated[int, operator.add]
    gate_passed: bool  # the gate's latest verdict


MAX_VALIDATION_RETRIES = 3

print(f"✅ EmailState fields: {list(EmailState.__annotations__)}")

✅ EmailState fields: ['topic', 'key_points', 'draft_email', 'final_email', 'retry_count', 'gate_passed']


---
## 🧱 Part 3: The Chain Steps and the Gate

Three nodes make the LLM calls that do the work. Between the first and second call sits the **gate**: a check on the intermediate result that decides whether the chain may continue.

A gate can be any check — a length rule, a schema, a unit test, or a second, cheap LLM call acting as a judge. A keyword rule ("does the text contain *request*, *deadline*, *support* …?") is tempting but leaky: philosophical key points happily mention "support" and "progress", so vague topics slip through on a lucky sample. Here the gate is a **temperature-0 judge** that returns a structured verdict: do these key points give a work email a concrete purpose (a request, decision, deadline or next step)?

The gate is split in two, the idiomatic LangGraph shape:
- **`check_key_points`** (a node) runs the judge and writes the verdict into state
- **`route_after_check`** (a pure function) reads the verdict and returns the next edge: `write_draft` (pass), `regenerate` (loop back to extraction) or `fail` (retries exhausted → a terminal node that reports why the chain stopped)

In [4]:
# ============================================================================
# THE GATE'S JUDGE: A deterministic model that returns a structured verdict
# ============================================================================
from pydantic import BaseModel, Field


class GateVerdict(BaseModel):
    """Verdict on the key points extracted in step 1."""

    actionable: bool = Field(
        description=(
            "True only if the key points give a business email a concrete purpose: "
            "a request, decision, deadline or next step for the recipient"
        )
    )
    reason: str = Field(description="One sentence explaining the verdict")


# temperature 0: the gate should judge the same text the same way every time
judge = get_openai_llm(model="gpt-4o-mini").with_structured_output(GateVerdict)

print("✅ Gate judge ready")

✅ Gate judge ready


In [5]:
# ============================================================================
# CHAIN STEPS: Extract → check → (route) → draft → polish, plus a failure node
# ============================================================================
def extract_key_points(state: EmailState) -> dict:
    """Step 1: extract key points from the topic."""
    attempt = state.get("retry_count", 0) + 1
    response = llm.invoke(f"List 3 key topics about: {state['topic']}")
    print(f"✅ Key points extracted (attempt {attempt}/{MAX_VALIDATION_RETRIES})")
    # Return +1; the operator.add reducer sums the visits.
    return {"key_points": response.content, "retry_count": 1}


def check_key_points(state: EmailState) -> dict:
    """Gate, part 1: ask the judge and store its verdict in state."""
    verdict = judge.invoke(
        "You check step 1 of an email-writing pipeline before step 2 drafts a "
        f"work email from it.\n\nKey points:\n{state['key_points']}"
    )
    icon = "✅" if verdict.actionable else "❌"
    print(f"{icon} Gate verdict: actionable={verdict.actionable} — {verdict.reason}")
    return {"gate_passed": verdict.actionable}


def route_after_check(state: EmailState) -> str:
    """Gate, part 2: pure routing on the stored verdict."""
    if state["gate_passed"]:
        return "write_draft"
    if state["retry_count"] >= MAX_VALIDATION_RETRIES:
        return "fail"
    print(f"🔄 Regenerating ({state['retry_count']}/{MAX_VALIDATION_RETRIES})")
    return "regenerate"


def fail_validation(state: EmailState) -> dict:
    """Terminal node: stop the chain instead of drafting from weak key points."""
    msg = (
        f"Stopped: could not extract actionable key points after "
        f"{state['retry_count']} attempts. "
        f"Topic may be too abstract: '{state['topic']}'"
    )
    print(f"🛑 {msg}")
    return {"final_email": msg}


def write_draft(state: EmailState) -> dict:
    """Step 2: write an email draft from the key points."""
    response = llm.invoke(
        f"Write a professional email draft covering these points: {state['key_points']}"
    )
    print("✅ Draft written")
    return {"draft_email": response.content}


def polish_email(state: EmailState) -> dict:
    """Step 3: polish the draft and add a greeting/closing."""
    response = llm.invoke(
        f"Polish this email and add proper greeting/closing: {state['draft_email']}"
    )
    print("✅ Email polished")
    return {"final_email": response.content}


print("✅ 5 nodes and 1 router defined")

✅ 5 nodes and 1 router defined


---
## 🏗️ Part 4: Building the Sequential Workflow

The happy path is plain `add_edge` calls. The gate's router is attached with `add_conditional_edges`: after `check_points` runs, LangGraph calls `route_after_check` and follows the edge named by its return value. The `regenerate` edge points *back* to `extract_points` — that one edge is what turns a straight chain into a bounded loop.

In [6]:
# ============================================================================
# BUILD THE GRAPH: Fixed edges for the chain, a conditional edge for the gate
# ============================================================================
from langgraph.graph import END, START, StateGraph

workflow = StateGraph(EmailState)

workflow.add_node("extract_points", extract_key_points)
workflow.add_node("check_points", check_key_points)
workflow.add_node("write_draft", write_draft)
workflow.add_node("polish_final", polish_email)
workflow.add_node("fail_validation", fail_validation)

workflow.add_edge(START, "extract_points")
workflow.add_edge("extract_points", "check_points")
workflow.add_conditional_edges(
    "check_points",
    route_after_check,
    {
        "write_draft": "write_draft",
        "regenerate": "extract_points",  # loop back for a fresh sample
        "fail": "fail_validation",
    },
)
workflow.add_edge("write_draft", "polish_final")
workflow.add_edge("polish_final", END)
workflow.add_edge("fail_validation", END)

compiled_workflow = workflow.compile()

print("✅ Workflow compiled")

✅ Workflow compiled


In [7]:
# ============================================================================
# INSPECT THE GRAPH: Print nodes and edges (no network call needed)
# ============================================================================
g = compiled_workflow.get_graph()

print("🧩 Nodes:", [n for n in g.nodes if not n.startswith("__")])
print("🔗 Edges:")
for e in g.edges:
    kind = "conditional" if e.conditional else "fixed"
    print(f"   {e.source:>18} ──▶ {e.target:<18} ({kind})")

🧩 Nodes: ['extract_points', 'check_points', 'write_draft', 'polish_final', 'fail_validation']
🔗 Edges:
            __start__ ──▶ extract_points     (fixed)
         check_points ──▶ extract_points     (conditional)
         check_points ──▶ fail_validation    (conditional)
         check_points ──▶ write_draft        (conditional)
       extract_points ──▶ check_points       (fixed)
          write_draft ──▶ polish_final       (fixed)
      fail_validation ──▶ __end__            (fixed)
         polish_final ──▶ __end__            (fixed)


Read the edges: three *conditional* edges leave `check_points` (the gate's three outcomes), and every other edge is fixed. That is the whole pattern — a fixed chain with one checkpoint.

---
## 🧪 Part 5: Running the Workflow

`create_email` invokes the graph with a `recursion_limit` as a second safety net: even if the gate logic had a bug, LangGraph would stop after that many steps instead of looping forever.

The first topic is concrete, so its key points should clear the gate on the first try. The second is abstract, so the gate should reject it until retries run out.

In [8]:
# ============================================================================
# RUNNER: Invoke the chain for one topic
# ============================================================================
def create_email(topic: str) -> dict:
    """Run the complete email creation workflow and return the final state."""
    print(f"📧 Creating email about: '{topic}'")
    print("-" * 60)
    result = compiled_workflow.invoke(
        {"topic": topic, "retry_count": 0},
        {"recursion_limit": MAX_VALIDATION_RETRIES * 4},
    )
    print(f"\n🎉 Workflow finished after {result['retry_count']} extraction attempt(s)")
    return result


print("✅ Runner ready")

✅ Runner ready


In [9]:
# ============================================================================
# TEST CASE 1: Concrete topic — should pass the gate and produce an email
# ============================================================================
result_1 = create_email("Request for project deadline extension due to technical challenges")
print(f"\n📄 FINAL EMAIL:\n{result_1['final_email']}")

📧 Creating email about: 'Request for project deadline extension due to technical challenges'
------------------------------------------------------------
✅ Key points extracted (attempt 1/3)
✅ Gate verdict: actionable=True — The key points provide a clear purpose for the email by outlining a request for a project deadline extension, justifying the need for it, and proposing a new timeline.
✅ Draft written
✅ Email polished

🎉 Workflow finished after 1 extraction attempt(s)

📄 FINAL EMAIL:
Subject: Request for Project Deadline Extension Due to Technical Challenges

Dear [Recipient's Name],

I hope this message finds you well. I am writing to formally request an extension on the deadline for the [Project Name] due to some significant technical challenges we have encountered.

**Identification of Technical Challenges**  
Throughout the course of the project, we have faced several unforeseen technical issues that have hindered our progress. Specifically, [briefly describe the technical chal

In [10]:
# ============================================================================
# TEST CASE 2: Abstract topic — should fail the gate and stop after 3 attempts
# ============================================================================
result_2 = create_email("The meaning of life and happiness")
print(f"\n📄 FINAL EMAIL:\n{result_2['final_email']}")

📧 Creating email about: 'The meaning of life and happiness'
------------------------------------------------------------
✅ Key points extracted (attempt 1/3)
❌ Gate verdict: actionable=False — The key points provide informative content but do not present a specific request, decision, or next step for the recipient.
🔄 Regenerating (1/3)
✅ Key points extracted (attempt 2/3)
❌ Gate verdict: actionable=False — The key points provide insights into philosophical and psychological topics but do not present a specific request, decision, or next step for the recipient.
🔄 Regenerating (2/3)
✅ Key points extracted (attempt 3/3)
❌ Gate verdict: actionable=False — The key points provide insights into philosophical and psychological concepts but do not present a specific request or action for the recipient.
🛑 Stopped: could not extract actionable key points after 3 attempts. Topic may be too abstract: 'The meaning of life and happiness'

🎉 Workflow finished after 3 extraction attempt(s)

📄 FINAL EMA

Compare the two runs: the first passed the gate on the first attempt and went on to draft and polish; the second was rejected on every attempt, never reached the draft or polish steps, and ended in `fail_validation`. The gate spent three cheap judge calls to avoid writing — and polishing — an email with no purpose.

---
## 📝 Summary

In this notebook, we learned:

### 1. The prompt chaining pattern
- A task split into fixed, focused LLM calls; each step's output is the next step's input
- Trades latency (more calls) for accuracy (each call is simpler)

### 2. Workflow state
- A `TypedDict` holds one field per step; nodes return only the keys they change
- A reducer (`Annotated[int, operator.add]`) turns a field into a counter

### 3. Quality gates
- A gate checks an intermediate output — with code, or with a cheap temperature-0 judge; keyword rules leak
- Idiomatic shape: a node computes the verdict into state, a pure router returns the next edge
- Failing early avoids spending tokens on steps that cannot succeed

### 4. Bounded regeneration
- A conditional edge back to an earlier node makes a loop; a counter plus a terminal `fail` node bounds it
- Regeneration only helps if the step can produce a different sample (temperature > 0)

### 5. Running and reading the chain
- `recursion_limit` is a second guard against runaway loops
- The final state records every intermediate value, which makes a chain easy to debug

### Next Steps
- **`02_prompt_chaining_structured_output.ipynb`**: pass validated Pydantic objects between the steps instead of raw strings